# Interactive Visual Analytics with Folium

This notebook completes the SpaceX launch-site location lab using the deterministic local dataset produced in Lab 3.

## Objectives

- Mark all launch sites on an interactive map.
- Mark successful and unsuccessful launches.
- Calculate and display distances to nearby geographic features.

In [1]:
from pathlib import Path
import folium
import pandas as pd
from folium.features import DivIcon
from folium.plugins import MarkerCluster, MousePosition

In [2]:
dataset_candidates=[Path('../03-data-wrangling/dataset_part_2.csv'),Path('03-data-wrangling/dataset_part_2.csv')]
dataset_path=next((p for p in dataset_candidates if p.exists()),None)
if dataset_path is None: raise FileNotFoundError('Could not find dataset_part_2.csv from Lab 3.')
spacex_df=pd.read_csv(dataset_path).rename(columns={'LaunchSite':'Launch Site','Latitude':'Lat','Longitude':'Long','Class':'class'})[['Launch Site','Lat','Long','class']]
print(f'Loaded {len(spacex_df)} launch records from {dataset_path.resolve()}')
spacex_df.head()

Loaded 90 launch records from K:\Uni Coding\ibm-data-science-capstone\03-data-wrangling\dataset_part_2.csv


,Launch Site,Lat,Long,class
0,CCAFS SLC 40,28.561857,-80.577366,0
1,CCAFS SLC 40,28.561857,-80.577366,0
2,CCAFS SLC 40,28.561857,-80.577366,0
3,VAFB SLC 4E,34.632093,-120.610829,0
4,CCAFS SLC 40,28.561857,-80.577366,0


## Task 1: Mark all launch sites on a map

In [3]:
launch_sites_df=spacex_df.groupby('Launch Site',as_index=False).first()[['Launch Site','Lat','Long']]
nasa_coordinate=[29.559684888503615,-95.0830971930759]
site_map=folium.Map(location=nasa_coordinate,zoom_start=5,control_scale=True)
folium.Circle(nasa_coordinate,radius=1000,color='#d35400',fill=True,popup='NASA Johnson Space Center').add_to(site_map)
folium.Marker(nasa_coordinate,icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0),html='<div style="font-size:12px;color:#d35400"><b>NASA JSC</b></div>')).add_to(site_map)
for _,row in launch_sites_df.iterrows():
    coordinate=[row['Lat'],row['Long']]
    folium.Circle(coordinate,radius=1000,color='#1f77b4',fill=True,popup=f"{row['Launch Site']} ({row['Lat']:.5f}, {row['Long']:.5f})").add_to(site_map)
    folium.Marker(coordinate,icon=DivIcon(icon_size=(150,20),icon_anchor=(0,0),html=f'<div style="font-size:12px;color:#1f4e79"><b>{row["Launch Site"]}</b></div>')).add_to(site_map)
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS SLC 40,28.561857,-80.577366
1,KSC LC 39A,28.608058,-80.603956
2,VAFB SLC 4E,34.632093,-120.610829


In [4]:
site_map

### Task 1 findings

The launch sites are in Florida and California. They are not all close to the Equator: their latitudes are approximately 28-35 degrees north. Each site is close to the coast, supporting safe downrange flight corridors and recovery operations over the ocean.

## Task 2: Mark successful and unsuccessful launches

In [5]:
spacex_df['marker_color']=spacex_df['class'].map({1:'green',0:'red'})
marker_cluster=MarkerCluster(name='Launch outcomes').add_to(site_map)
for _,record in spacex_df.iterrows():
    folium.Marker(location=[record['Lat'],record['Long']],popup=f"{record['Launch Site']} - {'Successful' if record['class']==1 else 'Unsuccessful'}",icon=folium.Icon(color='white',icon_color=record['marker_color'],icon='rocket',prefix='fa')).add_to(marker_cluster)
folium.LayerControl().add_to(site_map)
site_map

In [6]:
site_summary=(spacex_df.groupby('Launch Site')['class'].agg(launches='count',successful='sum').assign(failed=lambda x:x['launches']-x['successful'],success_rate=lambda x:x['successful']/x['launches']).reset_index())
site_summary

,Launch Site,launches,successful,failed,success_rate
0,CCAFS SLC 40,55,33,22,0.600000
1,KSC LC 39A,22,17,5,0.772727
2,VAFB SLC 4E,13,10,3,0.769231


The green and red clustered markers show launch outcomes by site. The summary table provides exact launch counts and success rates.

## Task 3: Calculate distances to nearby features

In [7]:
from math import atan2,cos,radians,sin,sqrt
def calculate_distance(lat1,lon1,lat2,lon2):
    R=6373.0
    lat1,lon1,lat2,lon2=map(radians,[lat1,lon1,lat2,lon2])
    dlon,dlat=lon2-lon1,lat2-lat1
    a=sin(dlat/2)**2+cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R*2*atan2(sqrt(a),sqrt(1-a))

The points below represent approximate closest coastline points identified while exploring the map. The same method applies to cities, railways, and highways.

In [8]:
coastline_points={'CCAFS SLC 40':(28.56200,-80.56700),'KSC LC 39A':(28.60800,-80.60400),'VAFB SLC 4E':(34.63280,-120.61080)}
site_distances=launch_sites_df.copy()
site_distances['coastline_lat']=site_distances['Launch Site'].map(lambda x:coastline_points[x][0])
site_distances['coastline_long']=site_distances['Launch Site'].map(lambda x:coastline_points[x][1])
site_distances['distance_to_coastline_km']=site_distances.apply(lambda r:calculate_distance(r['Lat'],r['Long'],r['coastline_lat'],r['coastline_long']),axis=1)
site_distances[['Launch Site','distance_to_coastline_km']]

,Launch Site,distance_to_coastline_km
0,CCAFS SLC 40,1.012813
1,KSC LC 39A,0.007808
2,VAFB SLC 4E,0.078684


In [9]:
formatter='function(num) {return L.Util.formatNum(num, 5);};'
MousePosition(position='topright',separator=' Long: ',empty_string='NaN',lng_first=False,num_digits=5,prefix='Lat:',lat_formatter=formatter,lng_formatter=formatter).add_to(site_map)
for _,row in site_distances.iterrows():
    start=[row['Lat'],row['Long']]; end=[row['coastline_lat'],row['coastline_long']]
    folium.Marker(end,popup=f"Approximate coastline point: {row['distance_to_coastline_km']:.2f} km from {row['Launch Site']}",icon=DivIcon(icon_size=(120,20),icon_anchor=(0,0),html=f'<div style="font-size:11px;color:#d35400"><b>{row["distance_to_coastline_km"]:.2f} km</b></div>')).add_to(site_map)
    folium.PolyLine([start,end],color='#d35400',weight=2,tooltip=f"{row['Launch Site']}: {row['distance_to_coastline_km']:.2f} km").add_to(site_map)
site_map

### Task 3 findings

The calculated distances show that these launch sites are close to the coastline. This is consistent with their coastal geography and the operational advantage of launching over the ocean.

In [10]:
output_path=Path('launch_site_location_map.html')
site_map.save(output_path)
print(f'Saved interactive map to {output_path.resolve()}')
assert len(launch_sites_df)==3
assert spacex_df['marker_color'].isin(['green','red']).all()
assert (site_distances['distance_to_coastline_km']>0).all()
print(f'Validated {len(spacex_df)} launches across {len(launch_sites_df)} launch sites.')

Saved interactive map to K:\Uni Coding\ibm-data-science-capstone\06-folium-map\launch_site_location_map.html
Validated 90 launches across 3 launch sites.
